In [2]:
import os
import subprocess
import logging
import numpy as np
import scipy.io as sio
import time
from obspy import read
from obspy import UTCDateTime
from datetime import datetime

# Helper: Convert UTC string to MATLAB datenum
def utc_to_datenum(dt_str):
    dt = datetime.strptime(dt_str, "%Y-%m-%d:%H:%M:%S.%f")
    return dt.toordinal() + 366 + (dt.hour + dt.minute / 60 + dt.second / 3600 + dt.microsecond / 3.6e9) / 24

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

def download_with_curl(start, end, network, station_pattern, location_pattern, channel_patterns, username, token, output_path):
    url = (
        "http://service.iris.edu/fdsnws/dataselect/1/queryauth"
        f"?network={network}"
        f"&station={station_pattern}"
        f"&location={location_pattern}"
        f"&channel={','.join(channel_patterns)}"
        f"&starttime={start}"
        f"&endtime={end}"
    )
    curl_cmd = [
        "curl", "--digest",
        "--user", f"{username}:{token}",
        url,
        "-o", output_path
    ]
    result = subprocess.run(curl_cmd, capture_output=True)
    if result.returncode != 0:
        logger.warning(f"curl failed: {result.stderr.decode().strip()}")
        
        #logger.info(f"Downloaded {output_path}")

def download_seismic_data_with_curl(
    start_datetime="2022-10-07T18:00:00",
    end_datetime="2023-10-07T00:00:00",
    network="2F",
    station_pattern="AX*",
    location_pattern="*",
    channel_patterns=["HH?", "EL?"],
    output_folder="data",
    output_foldermseed="datamseed",
    username="mczhang8@uw.edu",
    token="RxL42LH6XTNFybHb",
    increment_hours=1,
    overlap_seconds=15
):
    try:
        start = UTCDateTime(start_datetime)
        end = UTCDateTime(end_datetime)
        logger.info(f"Processing range: {start_datetime} to {end_datetime}")
    except Exception as e:
        logger.error(f"Invalid datetime format: {str(e)}")
        return {}

    increment_seconds = increment_hours * 3600
    overlap_seconds = float(overlap_seconds)
    current_date = start
    t0 = time.time()

    while current_date <= end:
        interval_start = current_date - overlap_seconds
        interval_end = current_date + increment_seconds + overlap_seconds
        date_str = current_date.strftime('%Y-%m-%d-%H-%M-%S')

        year = current_date.strftime('%Y')
        month = current_date.strftime('%m')
        output_dir = os.path.join(output_folder, year, month)
        output_dir_mseed = os.path.join(output_foldermseed, year, month)
        os.makedirs(output_dir_mseed, exist_ok=True)
        os.makedirs(output_dir, exist_ok=True)

        output_mseed = os.path.join(output_dir_mseed, f"{date_str}.mseed")
        output_mat = os.path.join(output_dir, f"{date_str}.mat")

        # Step 1: Use curl to download mseed file
        download_with_curl(
            start=interval_start.strftime("%Y-%m-%dT%H:%M:%S"),
            end=interval_end.strftime("%Y-%m-%dT%H:%M:%S"),
            network=network,
            station_pattern=station_pattern,
            location_pattern=location_pattern,
            channel_patterns=channel_patterns,
            username=username,
            token=token,
            output_path=output_mseed
        )

        # Step 2: Load and process with ObsPy
        try:
            stream = read(output_mseed)
            trace_dict = {
                'network': [], 'station': [], 'location': [], 'channel': [],
                'sensitivity': [], 'sensitivityFrequency': [], 'data': [],
                'sampleCount': [], 'sampleRate': [], 'startTime': [], 'endTime': []
            }

            for trace in stream:
                trace_dict['network'].append(trace.stats.network)
                trace_dict['station'].append(trace.stats.station)
                trace_dict['location'].append(trace.stats.location)
                trace_dict['channel'].append(trace.stats.channel)
                trace_dict['sensitivity'].append(np.nan)
                trace_dict['sensitivityFrequency'].append(np.nan)
                trace_dict['data'].append(trace.data.astype(np.float64))
                trace_dict['sampleCount'].append(int(trace.stats.npts))
                trace_dict['sampleRate'].append(float(trace.stats.sampling_rate))
                trace_dict['startTime'].append(trace.stats.starttime.strftime("%Y-%m-%d:%H:%M:%S.%f"))
                trace_dict['endTime'].append(trace.stats.endtime.strftime("%Y-%m-%d:%H:%M:%S.%f"))

                # logger.info(
                #     f"Processed {trace.id}: {trace.stats.npts} samples, "
                #     f"{trace.stats.sampling_rate} Hz, {trace.stats.starttime} to {trace.stats.endtime}"
                # )

            if trace_dict['network']:
                num_traces = len(trace_dict['network'])
                dtype = [
                    ('network', 'O'), ('station', 'O'), ('location', 'O'), ('channel', 'O'),
                    ('sensitivity', 'f8'), ('sensitivityFrequency', 'f8'), ('data', 'O'),
                    ('sampleCount', 'i4'), ('sampleRate', 'f8'),
                    ('startTime', 'f8'), ('endTime', 'f8')
                ]

                trace_struct = np.zeros(num_traces, dtype=dtype)
                for i in range(num_traces):
                    trace_struct[i] = (
                        trace_dict['network'][i],
                        trace_dict['station'][i],
                        trace_dict['location'][i],
                        trace_dict['channel'][i],
                        trace_dict['sensitivity'][i],
                        trace_dict['sensitivityFrequency'][i],
                        trace_dict['data'][i],
                        trace_dict['sampleCount'][i],
                        trace_dict['sampleRate'][i],
                        utc_to_datenum(trace_dict['startTime'][i]),
                        utc_to_datenum(trace_dict['endTime'][i])
                    )

                sio.savemat(output_mat, {'trace': trace_struct}, format='5', do_compression=True)
                file_size = os.path.getsize(output_mat) / 1024
                logger.info(f"Saved {output_mat} ({file_size:.2f} KB), {num_traces} traces")
                elapsed = time.time() - t0
                logger.info(f"⏱️ Total time: {elapsed:.2f} seconds")


        except Exception as e:
            logger.error(f"Failed to read or process {output_mseed}: {str(e)}")

        current_date += increment_seconds

# Run the script
if __name__ == "__main__":
    download_seismic_data_with_curl()


2025-05-21 11:06:21,619 - INFO - Processing range: 2022-10-07T18:00:00 to 2023-10-07T00:00:00
2025-05-21 11:06:35,751 - INFO - Saved data/2022/10/2022-10-07-18-00-00.mat (47733.25 KB), 45 traces
2025-05-21 11:06:35,752 - INFO - ⏱️ Total time: 14.13 seconds
2025-05-21 11:06:47,494 - INFO - Saved data/2022/10/2022-10-07-19-00-00.mat (46144.20 KB), 45 traces
2025-05-21 11:06:47,495 - INFO - ⏱️ Total time: 25.87 seconds
2025-05-21 11:06:59,572 - INFO - Saved data/2022/10/2022-10-07-20-00-00.mat (46883.90 KB), 45 traces
2025-05-21 11:06:59,573 - INFO - ⏱️ Total time: 37.95 seconds
2025-05-21 11:07:12,469 - INFO - Saved data/2022/10/2022-10-07-21-00-00.mat (48277.62 KB), 45 traces
2025-05-21 11:07:12,470 - INFO - ⏱️ Total time: 50.85 seconds
2025-05-21 11:07:26,369 - INFO - Saved data/2022/10/2022-10-07-22-00-00.mat (48354.97 KB), 45 traces
2025-05-21 11:07:26,370 - INFO - ⏱️ Total time: 64.75 seconds
2025-05-21 11:07:41,135 - INFO - Saved data/2022/10/2022-10-07-23-00-00.mat (45093.62 KB), 

KeyboardInterrupt: 